# DWD ICON precipitation — direct-HTTPS fetch

DWD Open Data publishes ICON forecasts as per-variable, bz2-compressed GRIB2 files over plain HTTPS (no SDK, no auth). earthlens' DWD centre downloads and decompresses them.

**Requirements** (live download):

```bash
pip install earthlens[nwp]
```

!!! note "Native grid"
    DWD's native ICON-global files are on an **icosahedral** grid, which is not a regular lat/lon raster — so this notebook fetches the raw GRIB2 rather than cropping it to a COG. For a croppable COG use a regular-lat/lon ICON product (e.g. `icon-eu`).

In [1]:
import datetime as dt
from pathlib import Path

from earthlens.nwp import Catalog
from earthlens.nwp.centres.dwd import DWDCentre

model = Catalog().get_model('icon-global')
model.bands

{'temperature_2m': 'T_2M',
 'precipitation_acc': 'TOT_PREC',
 'dewpoint_2m': 'TD_2M',
 'relative_humidity_2m': 'RELHUM_2M',
 'wind_u_10m': 'U_10M',
 'wind_v_10m': 'V_10M',
 'wind_gust': 'VMAX_10M',
 'pressure_msl': 'PMSL',
 'surface_pressure': 'PS',
 'total_cloud_cover': 'CLCT',
 'cape': 'CAPE_ML',
 'geopotential_height_1000hPa': 'FI@1000',
 'geopotential_height_925hPa': 'FI@925',
 'geopotential_height_850hPa': 'FI@850',
 'geopotential_height_700hPa': 'FI@700',
 'geopotential_height_600hPa': 'FI@600',
 'geopotential_height_500hPa': 'FI@500',
 'geopotential_height_400hPa': 'FI@400',
 'geopotential_height_300hPa': 'FI@300',
 'geopotential_height_250hPa': 'FI@250',
 'geopotential_height_200hPa': 'FI@200',
 'geopotential_height_150hPa': 'FI@150',
 'geopotential_height_100hPa': 'FI@100',
 'geopotential_height_50hPa': 'FI@50',
 'temperature_1000hPa': 'T@1000',
 'temperature_925hPa': 'T@925',
 'temperature_850hPa': 'T@850',
 'temperature_700hPa': 'T@700',
 'temperature_600hPa': 'T@600',
 'tem

In [2]:
# ICON-global runs 00/06/12/18Z and DWD keeps only ~the last day online,
# so pick the most recent run that is already published (~back off 7 h,
# floored to the 6-hourly cycle grid).
now = dt.datetime.now(dt.UTC).replace(tzinfo=None)
ref = now - dt.timedelta(hours=7)
cycle = ref.replace(hour=(ref.hour // 6) * 6, minute=0, second=0, microsecond=0)
cycle

datetime.datetime(2026, 5, 27, 6, 0)

In [3]:
out_dir = Path('out/icon')
out_dir.mkdir(parents=True, exist_ok=True)
grib_path = DWDCentre(out_dir).fetch_one(
    model, cycle, step=0, params=['precipitation_acc'], mirror='auto'
)
grib_path.name, grib_path.stat().st_size

('icon_2026052706_f000.grib2', 193)

The downloaded `.grib2` holds the requested band's decompressed messages. For a regular-grid model you would instead call `EarthLens(data_source="nwp", variables={...}).download()` and receive a bbox-cropped COG, exactly like the GFS quickstart.